# Nonplanar Spacer

A ring-shaped spacer where Z varies continuously as a sine wave around the circumference — a simple but effective non-planar test print.

Run cells top to bottom with shift+enter. If you change a cell, re-run it and all cells below it.

In [1]:
import fullcontrol as fc
from fullcontrol.devices.personal.tuchanka.setup import starting_steps, ending_steps, base_settings, load_profile, next_output_path
from math import cos, tau

In [2]:
# printer / startup parameters

design_name  = 'nonplanar_spacer'
initial_tool = 3
tools_used   = [3]
print_area   = (20, 20, 120, 120)  # (x_min, y_min, x_max, y_max) mm

profile = load_profile('jessie_silky_pla_0.8')  # swap name to change material

startup = starting_steps(
    initial_tool=initial_tool,
    tools_used=tools_used,
    first_layer_temps=profile['first_layer_temps'],
    idle_temps=profile['idle_temps'],
    mbl_temp=profile['mbl_temp'],
    bed_temp=profile['bed_temp'],
    travel_speed=profile['travel_speed'],
    zhop=profile['zhop'],
    print_area=print_area,
    heat_soak=False,
    nozzle_clean=False,
)

In [3]:
# design parameters — pulled from profile, override here if needed

EW          = profile['extrusion_width']
EH          = profile['extrusion_height']
print_speed = profile['print_speed']

# spacer geometry
waves              = 6      # number of waves around the circumference
total_thickness    = 4      # mm — total thickness of the spacer
D1                 = 8      # mm — inner hole diameter
D_ratio            = 3      # outer diameter = D1 * D_ratio
EW_EH_ratio        = 2      # extrusion width to height ratio (2 recommended)
overlap_percent    = 20     # lateral overlap between adjacent passes (%)
contraction_factor = 1.2    # pulls wave tips inward to prevent nozzle collision
quantity           = 1      # number of copies to print

In [4]:
# generate design steps

EW = EH * EW_EH_ratio
D2 = D1 * D_ratio
overlap = (overlap_percent / 100) * EW
height = total_thickness - EH
R1, R2 = D1 / 2, D2 / 2
rings = int((R2 - R1) / (EW - overlap))
segs_per_ring = (waves * 2) * int(128 / (waves * 2))

centre = fc.Point(x=0, y=0, z=0)

design_steps = [
    fc.move_polar(centre, centre, R2, (0.5 + ((waves + 1) % 2) / (2 * waves)) * tau),
    fc.Extruder(on=False),
    centre,
    fc.Extruder(on=True),
]

# spiral purge / flow stabiliser inside the hole (if space allows)
purge_spiral_passes = min(int((R1 - EW) / EW) - 1, 3)
if purge_spiral_passes > 0:
    design_steps.extend(fc.spiralXY(centre, EW / 2, R1 - EW, 0, purge_spiral_passes, 200))

# spiral outward ring by ring, varying Z sinusoidally
for ring in range(rings):
    for seg in range(segs_per_ring + 1):
        angle_now  = (seg / segs_per_ring) * tau
        z_now      = height * (ring / (rings - 1)) * (0.5 - 0.5 * cos(angle_now * waves))
        radius_now = R1 + EW / 2 + ring * (EW - overlap) - (z_now * contraction_factor)
        centre.z   = z_now
        design_steps.append(fc.polar_to_point(centre, radius_now, angle_now))

# print multiple copies side by side
if quantity > 1:
    design_steps = fc.move(design_steps, fc.Vector(x=R2 * 2 + 5), copy=True, copy_quantity=quantity)

# centre the design on the bed — offset z by 0.8*EH for first-layer squish
bed_cx = (print_area[0] + print_area[2]) / 2
bed_cy = (print_area[1] + print_area[3]) / 2
model_offset = fc.Vector(x=bed_cx, y=bed_cy, z=0.8 * EH)
design_steps = fc.move(design_steps, model_offset)

In [5]:
# preview

annotations = [
    fc.PlotAnnotation(point=design_steps[0], label='Start'),
    fc.PlotAnnotation(point=design_steps[-1], label='End'),
]

fc.transform(
    design_steps + annotations,
    'plot',
    fc.PlotControls(
        color_type='print_sequence',
        initialization_data={'extrusion_width': EW, 'extrusion_height': EH},
    ),
)

    - use fc.transform(..., controls=fc.PlotControls(style='tube') to disable this message or style='line' for a simpler line preview



In [ ]:
# generate and save gcode

shutdown = ending_steps(travel_speed=profile['travel_speed'], present_print=True)
steps = startup + design_steps + shutdown

gcode_controls = fc.GcodeControls(
    printer_name='custom',
    save_as=next_output_path(design_name, output_dir='gcode'),
    include_date=False,
    initialization_data=base_settings(
        print_speed=print_speed,
        extrusion_width=EW,
        extrusion_height=EH,
        nozzle_temp=profile['first_layer_temps'][initial_tool],
        bed_temp=profile['bed_temp'],
    ),
)
gcode = fc.transform(steps, 'gcode', gcode_controls)

---
*Tuchanka notebook — Petricore Labs*